In [ ]:
# Importar librerías estándar para análisis de datos y visualización
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Herramientas de Scikit-learn: escalado y métricas de regresión
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score

In [ ]:
# TensorFlow / Keras para construir la red neuronal recurrente
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LSTM, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import timeseries_dataset_from_array

In [ ]:
# Fijar semillas para reproducibilidad de resultados
np.random.seed(42)
tf.random.set_seed(42)

### Análisis Exploratorio

In [ ]:
# Ruta al dataset de ventas (serie temporal mensual)
file_path = '~/tensorflow_ejemplos/DATA/daily_climate.csv'

In [ ]:
# Cargar el CSV usando la columna DATE como índice de tipo datetime
df = pd.read_csv(file_path, index_col='date', parse_dates=True)
df.head()

In [ ]:
# Renombrar la columna a un nombre más descriptivo
df = df[['meantemp']]

In [ ]:
# Visualizar la serie temporal completa
df.plot(figsize=(12,6));

### Escalado y Train-Test Split

In [ ]:
# Usar los últimos 18 meses como conjunto de prueba (aproximadamente 1.5 años)
test_percent = 0.3
test_size = int(len(df) * test_percent)
test_ind = len(df) - test_size

In [ ]:
# Dividir la serie en entrenamiento y prueba sin mezclar (crítico en series temporales)
train = df.iloc[:test_ind].copy()
test = df.iloc[test_ind:].copy()

In [ ]:
# Escalar al rango [0, 1] usando únicamente el conjunto de entrenamiento
# En series temporales es vital no filtrar información del futuro al escalador
scaler = MinMaxScaler()
scaled_train = scaler.fit_transform(train)
scaled_test = scaler.transform(test)

### Secuencias para la RNN

In [ ]:
# Longitud de la ventana temporal: usar 12 meses para predecir el siguiente
length = 365
batch_size = 16
n_features = 1

In [ ]:
# Crear datasets de series temporales con la API moderna de Keras
# Para test no podemos truncar porque quedarían menos registros que la ventana.
train_dataset = timeseries_dataset_from_array(
    data=scaled_train[:-length],
    targets=scaled_train[length:].flatten(),
    sequence_length=length,
    batch_size=batch_size,
    shuffle=True
)

test_dataset = timeseries_dataset_from_array(
    data=scaled_test,
    targets=scaled_test[length:].flatten(),
    sequence_length=length,
    batch_size=batch_size
)

### Entrenamiento del Modelo

In [ ]:
# Definir arquitectura de la RNN usando la API Sequential moderna de Keras
model = Sequential([
    Input(shape=(length, n_features)),         # Capa de entrada: (ventana temporal, features)
    LSTM(units=128),    # Capa recurrente con 32 unidades
    Dense(units=1)
])

In [ ]:
# Compilar el modelo: MSE es la loss estándar para problemas de regresión
model.compile(optimizer='adam', loss='huber', metrics=['mae'])

In [ ]:
# EarlyStopping evita overfitting deteniendo el entrenamiento si val_loss no mejora
# ReduceLROnPlateau ajusta la tasa de aprendizaje cuando el entrenamiento se estanca
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

In [ ]:
# Entrenar el modelo; test_dataset se usa como validación para monitorear overfitting
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    callbacks=[early_stop],
    epochs=200
)

### Evaluación

In [ ]:
# Convertir el historial de entrenamiento a DataFrame para graficar
history_df = pd.DataFrame(history.history)

In [ ]:
# Graficar la evolución de la pérdida en entrenamiento y validación
history_df[['loss', 'val_loss']].plot();

### Predicciones sobre el conjunto de prueba (Walk-Forward)

In [ ]:
# Generar predicciones iterativas (walk-forward) sobre el conjunto de prueba
# Se usa el último batch de entrenamiento como semilla y se predice paso a paso
test_predictions = []
first_eval_batch = scaled_train[-length:]
current_batch = first_eval_batch.reshape((1, length, n_features))

for i in range(len(scaled_test)):
    current_prediction = model.predict(current_batch, verbose=0)
    test_predictions.append(current_prediction[0, 0])
    current_batch = np.concatenate(
        [current_batch[:, 1:, :], current_prediction.reshape(1, 1, 1)], axis=1)

In [ ]:
# Convertir las predicciones escaladas de vuelta a la escala original
test_predictions = np.array(test_predictions).reshape(-1, 1)
predictions = scaler.inverse_transform(test_predictions)

# Añadir las predicciones al DataFrame de prueba para visualización comparativa
test['Predictions'] = predictions.flatten()

In [ ]:
# Graficar valores reales vs predicciones en el conjunto de prueba
test.plot(figsize=(12,6));

In [ ]:
# Error Absoluto Medio (MAE): interpretable en las mismas unidades de ventas
y_test = test['meantemp'].values
mean_absolute_error(y_test, predictions)

In [ ]:
# Raíz del Error Cuadrático Medio (RMSE): penaliza más los errores grandes
np.sqrt(mean_squared_error(y_test, predictions))

In [ ]:
# Explained Variance Score: 1.0 indica predicción perfecta
explained_variance_score(y_test, predictions)

### Forecast Futuro

In [ ]:
# IMPORTANTE: reutilizar el mismo scaler ajustado en entrenamiento
# Escalar la serie completa con los parámetros aprendidos del train
full_scaled = scaler.transform(df)

In [ ]:
# Generar predicciones futuras más allá del último dato conocido (2019-10-01)
forecast = []
periods = 90
first_eval_batch = full_scaled[-length:]
current_batch = first_eval_batch.reshape((1, length, n_features))

for i in range(periods):
    current_prediction = model.predict(current_batch, verbose=0)
    forecast.append(current_prediction[0, 0])
    current_batch = np.concatenate(
        [current_batch[:, 1:, :], current_prediction.reshape(1, 1, 1)], axis=1)

In [ ]:
# Convertir las predicciones futuras a la escala original de ventas
forecast = np.array(forecast).reshape(-1, 1)
forecast = scaler.inverse_transform(forecast)
forecast

In [ ]:
# El último dato es de octubre 2019; el forecast comienza en noviembre 2019
# Frecuencia 'MS' = Month Start (primer día de cada mes)
forecast_index = pd.date_range(start='2017-04-24', periods=periods, freq='D')

In [ ]:
# Crear DataFrame con las predicciones futuras
forecast_df = pd.DataFrame(data=forecast, index=forecast_index, columns=['Forecast'])
forecast_df

In [ ]:
# Visualizar el histórico junto con el forecast futuro
ax = df.plot(figsize=(12,6))
forecast_df.plot(ax=ax);